> **Note (2026-06-12):** the raw component parquets this notebook evaluates no longer
> live in a local `data/` directory. They are on S3 in dated pull snapshots:
> `s3://sagemaker-us-east-1-209479286572/datasets/QuantTradingModelData/raw/<label>/`
> (legacy 2000–2025 pull: `raw/20251229/`; produced by `GetFMPData/fmp_data_pull.py`).
> Read cells below have been repointed; legacy local-write cells are kept for reference only.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
import requests
import numpy as np
import pandas as pd


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)


def getFMPData(endpoint, urlhead=None, apikey=None, **params):
    if urlhead is None:
        urlhead = "https://financialmodelingprep.com/stable/"
    if apikey is None:
        apikey = os.getenv("FMP_API_KEY")

    if "from_" in params:
        params["from"] = params.pop("from_")

    url = urlhead + endpoint
    param_lst = [f"apikey={apikey}"] + [str(k) + "=" + str(v) for k, v in params.items()]
    url = "?".join([url, "&".join(param_lst)])
    data = requests.get(url).json()
    return data

In [19]:
!ls

DataEval.ipynb	  __init__.py	  test_data_AAPL_dailyprice.parquet
FMPDataClient.py  __pycache__
FMPDataPull.py	  test_data.json


In [213]:
df = pd.read_parquet("test_data_adjusteddailyprice.parquet")
df['date'] = pd.to_datetime(df['date'])
df.shape

(1455357, 7)

In [214]:
df

,symbol,date,adjOpen,adjHigh,adjLow,adjClose,volume
0,BTMD,2024-12-31,5.88,6.32,5.88,6.18,162100
1,BTMD,2024-12-30,5.96,5.96,5.76,5.91,53500
2,BTMD,2024-12-27,6.21,6.21,5.82,5.99,65622
3,BTMD,2024-12-26,5.91,6.19,5.85,6.17,72126
4,BTMD,2024-12-24,5.79,5.96,5.68,5.96,31300
...,...,...,...,...,...,...,...
1455352,DBRG-PI,2017-06-01,12.70,12.71,12.65,12.69,153290
1455353,DBRG-PI,2017-05-31,12.70,12.72,12.63,12.70,412325
1455354,DBRG-PI,2017-05-30,12.68,12.72,12.62,12.67,647810
1455355,DBRG-PI,2017-05-26,12.68,12.70,12.62,12.65,335140


In [215]:
df['symbol'].nunique()

960

In [129]:
v1 = df.groupby('symbol')['date'].agg(['min', 'max', 'count'])
# v1

In [130]:
df2 = pd.read_parquet("test_data_unadjusteddailyprice.parquet")
df2['date'] = pd.to_datetime(df2['date'])
df2.shape

(177159, 7)

In [131]:
df2

,symbol,date,adjOpen,adjHigh,adjLow,adjClose,volume
0,FLGC,2024-12-31,1.02,1.06,0.99,1.03,337818.0
1,FLGC,2024-12-30,1.04,1.05,0.98,1.03,460629.0
2,FLGC,2024-12-27,1.10,1.11,1.00,1.02,486486.0
3,FLGC,2024-12-26,1.02,1.14,1.02,1.11,289107.0
4,FLGC,2024-12-24,1.06,1.07,1.00,1.05,188292.0
...,...,...,...,...,...,...,...
177154,MXIM,2016-01-08,34.53,34.68,32.21,32.33,8204104.0
177155,MXIM,2016-01-07,34.49,34.57,33.81,34.26,5323510.0
177156,MXIM,2016-01-06,35.74,35.99,35.05,35.18,4406419.0
177157,MXIM,2016-01-05,36.78,37.32,36.19,36.23,3423675.0


In [132]:
df2['symbol'].nunique()

124

In [133]:
v2 = df2.groupby('symbol')['date'].agg(['min', 'max', 'count'])
# v2

In [134]:
pd.merge(v1, v2, left_index=True, right_index=True, how='left')

,min_x,max_x,count_x,min_y,max_y,count_y
symbol,,,,,,
AAIC-PB,2017-05-16,2023-12-13,1657,2017-05-16,2023-12-13,1657
ACB,2016-01-04,2024-12-31,2264,2016-01-04,2024-12-31,2264
ACW,2016-01-04,2016-11-17,223,2016-01-04,2016-11-17,223
ADAG,2021-02-09,2024-12-31,980,2021-02-09,2024-12-31,980
AIEV,2024-06-18,2024-12-31,136,2024-06-18,2024-12-31,136
AIXI,2023-03-09,2024-12-31,457,2023-03-09,2024-12-31,457
ALOG,2016-01-04,2018-06-29,628,2016-01-04,2018-06-29,628
AMPX,2022-09-16,2024-12-31,576,2022-09-16,2024-12-31,576
ANF,2016-01-04,2024-12-31,2264,2016-01-04,2024-12-31,2264


In [135]:
df3 = pd.read_parquet("test_data_incomestatement.parquet")
df3['date'] = pd.to_datetime(df3['date'])
df3.shape

(7113, 39)

In [136]:
df3

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,revenue,costOfRevenue,grossProfit,researchAndDevelopmentExpenses,generalAndAdministrativeExpenses,sellingAndMarketingExpenses,sellingGeneralAndAdministrativeExpenses,otherExpenses,operatingExpenses,costAndExpenses,netInterestIncome,interestIncome,interestExpense,depreciationAndAmortization,ebitda,ebit,nonOperatingIncomeExcludingInterest,operatingIncome,totalOtherIncomeExpensesNet,incomeBeforeTax,incomeTaxExpense,netIncomeFromContinuingOperations,netIncomeFromDiscontinuedOperations,otherAdjustmentsToNetIncome,netIncome,netIncomeDeductions,bottomLineNetIncome,eps,epsDiluted,weightedAverageShsOut,weightedAverageShsOutDil
0,2025-09-30,FLGC,USD,0001790169,2025-11-05,2025-11-05 17:06:11,2025,Q3,9748000.0,9413000.0,335000.0,0.0,2737000.0,0.0,2737000.0,864000.0,3601000.0,13014000.0,-61000.0,0.0,61000.0,201000.0,-3347000.0,-3548000.0,282000.0,-3266000.0,-343000.0,-3609000.0,18000.0,-3627000.0,-3032000.0,0.0,-6659000.0,0.0,-6659000.0,-12.1050,-12.1050,550102.0,550102.0
1,2025-06-30,FLGC,USD,0001790169,2025-08-01,2025-08-01 16:02:24,2025,Q2,14796000.0,11966000.0,2830000.0,73000.0,4003000.0,945000.0,4948000.0,375000.0,5396000.0,17362000.0,0.0,0.0,0.0,195000.0,-2112000.0,-2307000.0,-259000.0,-2566000.0,106000.0,-2460000.0,-48000.0,-2412000.0,0.0,0.0,-2412000.0,0.0,-2412000.0,-4.3800,-4.3800,550102.0,550102.0
2,2025-03-31,FLGC,USD,0001790169,2025-05-13,2025-05-13 16:56:00,2025,Q1,11786000.0,8898000.0,2888000.0,113000.0,3293000.0,905000.0,4198000.0,-446000.0,3865000.0,12763000.0,-51000.0,0.0,51000.0,182000.0,-487000.0,-669000.0,-308000.0,-977000.0,257000.0,-720000.0,38000.0,-758000.0,0.0,0.0,-758000.0,0.0,-758000.0,-3.9800,-3.9800,190640.0,190640.0
3,2024-12-31,FLGC,USD,0001790169,2025-03-24,2025-03-24 16:01:52,2024,Q4,13326000.0,10689000.0,2637000.0,191000.0,5986000.0,1863000.0,7849000.0,527000.0,8567000.0,19256000.0,-189000.0,0.0,189000.0,211000.0,-5782000.0,-5993000.0,63000.0,-5930000.0,-252000.0,-6182000.0,-106000.0,-6076000.0,0.0,0.0,-6130000.0,0.0,-6130000.0,-1.3500,-1.3500,4553320.0,4553320.0
4,2024-09-30,FLGC,USD,0001790169,2024-11-13,2024-11-13 16:35:48,2024,Q3,12465000.0,9626000.0,2839000.0,118000.0,4129000.0,1206000.0,5335000.0,1078000.0,6531000.0,16157000.0,-27000.0,0.0,27000.0,226000.0,-3711000.0,-3937000.0,245000.0,-3692000.0,-272000.0,-3964000.0,-164000.0,-3800000.0,0.0,0.0,-3773000.0,0.0,-3773000.0,-0.8300,-0.8300,4553320.0,4553320.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7108,1992-09-30,MXIM,USD,0000743316,1992-09-30,1992-09-29 20:00:00,1993,Q1,25000000.0,9400000.0,15600000.0,0.0,0.0,0.0,8600000.0,1100000.0,9700000.0,19100000.0,0.0,0.0,0.0,1100000.0,7300000.0,0.0,0.0,5900000.0,300000.0,6200000.0,2200000.0,4000000.0,0.0,0.0,4000000.0,0.0,4000000.0,0.0200,0.0200,200000000.0,200000000.0
7109,1992-06-30,MXIM,USD,0000743316,1992-06-30,1992-06-29 20:00:00,1992,Q4,23300000.0,8600000.0,14700000.0,0.0,0.0,0.0,8200000.0,1100000.0,9300000.0,17900000.0,0.0,0.0,0.0,1100000.0,6800000.0,0.0,0.0,5400000.0,300000.0,5700000.0,2000000.0,3700000.0,0.0,0.0,3700000.0,0.0,3700000.0,0.0211,0.0211,175000000.0,175000000.0
7110,1992-03-31,MXIM,USD,0000743316,1992-03-31,1992-03-30 19:00:00,1992,Q3,22100000.0,8200000.0,13900000.0,0.0,0.0,0.0,7200000.0,1300000.0,8500000.0,16700000.0,0.0,0.0,0.0,1300000.0,6700000.0,5400000.0,0.0,5400000.0,0.0,5400000.0,1900000.0,3500000.0,0.0,0.0,3500000.0,0.0,3500000.0,0.0200,0.0200,175000000.0,175000000.0
7111,1991-12-31,MXIM,USD,0000743316,1991-12-31,1991-12-30 19:00:00,1992,Q2,21100000.0,8100000.0,13000000.0,0.0,0.0,0.0,6800000.0,1200000.0,8000000.0,16100000.0,0.0,0.0,0.0,1200000.0,6300000.0,0.0,0.0,5000000.0,100000.0,5100000.0,1800000.0,3300000.0,0.0,0.0,3300000.0,0.0,3300000.0,0.0200,0.0200,165000000.0,165000000.0


In [137]:
df3['symbol'].nunique()

126

In [138]:
v3 = df3.groupby('symbol')['date'].agg(['min', 'max'])
v3

,min,max
symbol,,
AAIC-PB,1996-03-31,2023-09-30
ACB,2007-09-30,2025-09-30
ACW,2002-12-31,2016-09-30
ADAG,2019-03-31,2025-06-30
AIEV,2022-03-31,2025-09-30
AIXI,2021-06-30,2024-12-31
ALOG,1988-07-31,2018-04-30
AMPX,2021-03-31,2025-09-30
ANF,1996-04-30,2025-11-01


In [141]:
_v = pd.merge(v3, v1, left_index=True, right_index=True, how='outer')
# v3['indicator'] = (v3['incmMin'] < '2008-01-01').astype(int)

_v[_v.isna().any(axis=1)]

,min_x,max_x,min_y,max_y,count
symbol,,,,,
BRCB,2025-09-30,2025-09-30,NaT,NaT,NaN
BRIA,NaT,NaT,2024-11-27,2024-12-31,23.0
DSYWW,NaT,NaT,2024-06-06,2024-12-31,142.0
NKLR,2023-06-30,2025-09-30,NaT,NaT,NaN
RJF-PA,1995-12-29,2025-09-30,NaT,NaT,NaN
TTRX,2024-06-30,2025-06-30,NaT,NaT,NaN


In [142]:
# v3[(v3['indicator']==1) & v3['priceMin'].isna()]

In [190]:
price = getFMPData('historical-price-eod/dividend-adjusted', symbol='BRCB', from_='2000-01-01', to='2025-12-31')

In [191]:
pd.DataFrame(price)

,symbol,date,adjOpen,adjHigh,adjLow,adjClose,volume
0,BRCB,2025-12-26,22.83,23.67,22.83,23.35,207825
1,BRCB,2025-12-24,22.93,23.68,22.77,23.34,129808
2,BRCB,2025-12-23,23.42,23.81,22.25,23.03,447700
3,BRCB,2025-12-22,23.11,24.23,22.87,23.64,659412
4,BRCB,2025-12-19,22.28,22.85,21.73,22.69,3413241
5,BRCB,2025-12-18,21.76,22.57,21.67,22.34,519032
6,BRCB,2025-12-17,21.33,22.02,21.20,21.45,595700
7,BRCB,2025-12-16,22.17,22.61,20.87,21.18,401764
8,BRCB,2025-12-15,23.42,23.72,22.21,22.35,303400
9,BRCB,2025-12-12,24.02,24.52,23.24,23.35,263526


In [154]:
price2 = getFMPData('historical-price-eod/non-split-adjusted', symbol='DSYWW', from_='2000-01-01', to='2025-12-31')
pd.DataFrame(price2)

,symbol,date,adjOpen,adjHigh,adjLow,adjClose,volume
0,DSYWW,2025-12-24,0.02000,0.02920,0.02000,0.02000,5173
1,DSYWW,2025-12-23,0.02040,0.02920,0.02000,0.02400,4933
2,DSYWW,2025-12-22,0.02940,0.02940,0.02230,0.02940,1814
3,DSYWW,2025-12-19,0.02050,0.02950,0.02000,0.02950,46791
4,DSYWW,2025-12-18,0.02960,0.03000,0.02000,0.03000,6502
...,...,...,...,...,...,...,...
379,DSYWW,2024-06-12,0.12010,0.13000,0.09100,0.10000,98035
380,DSYWW,2024-06-11,0.16000,0.16000,0.08500,0.12100,314111
381,DSYWW,2024-06-10,0.15100,0.19000,0.12000,0.13490,257836
382,DSYWW,2024-06-07,0.07000,0.11000,0.03750,0.07950,119656


In [155]:
incm_stmt = getFMPData('income-statement', symbol='DSYWW', limit=120, period='quarter')
pd.DataFrame(incm_stmt)

""


In [156]:
incm_stmt

[]

In [161]:
bal_sheet = getFMPData('balance-sheet-statement', symbol='DSYWW', limit=120, period='quarter')
pd.DataFrame(bal_sheet)

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,cashAndCashEquivalents,shortTermInvestments,cashAndShortTermInvestments,netReceivables,accountsReceivables,otherReceivables,inventory,prepaids,otherCurrentAssets,totalCurrentAssets,propertyPlantEquipmentNet,goodwill,intangibleAssets,goodwillAndIntangibleAssets,longTermInvestments,taxAssets,otherNonCurrentAssets,totalNonCurrentAssets,otherAssets,totalAssets,totalPayables,accountPayables,otherPayables,accruedExpenses,shortTermDebt,capitalLeaseObligationsCurrent,taxPayables,deferredRevenue,otherCurrentLiabilities,totalCurrentLiabilities,longTermDebt,capitalLeaseObligationsNonCurrent,deferredRevenueNonCurrent,deferredTaxLiabilitiesNonCurrent,otherNonCurrentLiabilities,totalNonCurrentLiabilities,otherLiabilities,capitalLeaseObligations,totalLiabilities,treasuryStock,preferredStock,commonStock,retainedEarnings,additionalPaidInCapital,accumulatedOtherComprehensiveIncomeLoss,otherTotalStockholdersEquity,totalStockholdersEquity,totalEquity,minorityInterest,totalLiabilitiesAndTotalEquity,totalInvestments,totalDebt,netDebt
0,2024-06-30,DSYWW,USD,0001999297,2024-10-25,2024-10-25 16:05:45,2024,Q4,748099,0,748099,424052,47753,376299,600876,169008,13760,1955795,4146760,0,2362417,2362417,0,502965,0,7012142,0,8967937,2907582,430510,2477072,139611,0,190023,851836,3776879,60108,7074203,2401097,335241,1376046,0,2360147,6472531,0,525264,13546734,0,0,5708,-4898117,0,313612,0,-4578797,-4578797,0,8967937,0,2926361,2178262
1,2024-03-31,DSYWW,USD,0001999297,2024-06-12,2024-06-12 17:16:17,2024,Q3,31453,0,31453,47753,47753,0,0,83902,-47753,115355,0,0,2362417,2362417,35627550,0,-2362417,35627550,0,35742905,229000,0,229000,0,1570000,0,229000,0,878836,2677838,0,0,0,0,2012499,2012499,0,525264,4690338,0,0,35354043,-4301476,0,0,0,31052567,31052567,0,35742905,35627550,1570000,1538547
2,2023-12-31,DSYWW,USD,0001999297,2023-12-31,2023-12-30 19:00:00,2024,Q2,1121803,0,1121803,457397,20315,437082,1325975,226117,18197,3149489,4606257,0,79940,79940,0,515885,1323686,6525768,0,9675257,1567308,733079,834229,163161,1408471,0,427021,3728785,222612,7090337,0,450883,3521176,0,0,3972059,0,624632,11062396,0,0,50000,-3398283,1619428,341716,0,-1387139,-1387139,0,9675257,0,1408471,286668
3,2023-06-30,DSYWW,USD,0001999297,2023-06-30,2023-06-29 20:00:00,2023,Q4,3190995,0,3190995,1319183,211356,1107827,626196,1089117,33846,6259337,4686941,0,2509,2509,0,381191,752955,5823596,0,12082933,2162084,683424,1478660,116694,1408471,174650,154482,3385764,-1018610,6229053,0,500262,4765785,0,0,5266047,0,674912,11495100,0,0,5000,-3979667,4244352,318148,0,587833,587833,0,12082933,0,2083383,-1107612


In [179]:
cash_flow = getFMPData('cash-flow-statement', symbol='TTRX', limit=120, period='quarter')
pd.DataFrame(cash_flow)

""


In [182]:
_df = pd.read_parquet("test_data_keymetrics.parquet")
_df['date'] = pd.to_datetime(_df['date'])
_df.shape

(7181, 47)

In [183]:
_df['symbol'].nunique()

128

In [173]:
symbols = _df['symbol'].drop_duplicates().tolist()

In [174]:
# symbols_all = symbols.copy()

In [175]:
[x for x in symbols_all if x not in symbols]

['BRCB', 'DSYWW', 'BRIA', 'TTRX']

In [206]:
infotypes = [
    'adjusteddailyprice',
    'unadjusteddailyprice',
    'balancesheet',
    'cashflow',
    'incomestatement',
    'keymetrics',
    'enterprisevalues',
]

for i, infotype in enumerate(infotypes):
    path = f"test_data_{infotype}.parquet"
    df = pd.read_parquet(path)
    df['date'] = pd.to_datetime(df['date'])
    ser = df.groupby('symbol')['date'].count()
    ser.name = infotype

    if i == 0:
        left = ser.copy()
    else:
        left = pd.merge(left, ser, how='outer', left_index=True, right_index=True)

left = left.fillna(0)

In [207]:
left[(left == 0).any(axis=1)]

,adjusteddailyprice,unadjusteddailyprice,balancesheet,cashflow,incomestatement,keymetrics,enterprisevalues
symbol,,,,,,,
AAPG,0.0,0.0,17,19.0,19.0,19.0,19.0
APO-PB,0.0,0.0,66,68.0,70.0,70.0,70.0
BGMS,0.0,0.0,44,44.0,44.0,8.0,8.0
CAI,0.0,0.0,63,68.0,68.0,68.0,68.0
LZMH,0.0,0.0,5,0.0,0.0,5.0,5.0
MMA,192.0,192.0,1,0.0,1.0,0.0,0.0
MSIF,0.0,0.0,34,36.0,37.0,37.0,37.0
OG,418.0,418.0,6,0.0,8.0,9.0,9.0
OSRH,0.0,0.0,15,14.0,19.0,19.0,19.0


In [45]:
symbol = 'APO-PB'

price1 = getFMPData('historical-price-eod/dividend-adjusted', symbol=symbol, from_='2016-01-01', to='2024-12-31')
price2 = getFMPData('historical-price-eod/non-split-adjusted', symbol=symbol, from_='2016-01-01', to='2024-12-31')
bal = getFMPData('balance-sheet-statement', symbol=symbol, limit=120, period='quarter')
incm = getFMPData('income-statement', symbol=symbol, limit=120, period='quarter')
cf = getFMPData('cash-flow-statement', symbol=symbol, limit=120, period='quarter')
ev = getFMPData('enterprise-values', symbol=symbol, limit=120, period='quarter')
km = getFMPData('key-metrics', symbol=symbol, limit=120, period='quarter')
len(price1), len(price2), len(bal), len(cf), len(incm), len(km), len(ev)

(0, 0, 66, 68, 70, 70, 70)

In [209]:
getFMPData('cash-flow', symbol='AAPG', limit=120, period='quarter')

[]

In [210]:
cashflow_df = pd.read_parquet("test_data_cashflow.parquet")
cashflow_df[cashflow_df['symbol'] == 'AAPG']

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,netIncome,depreciationAndAmortization,deferredIncomeTax,stockBasedCompensation,changeInWorkingCapital,accountsReceivables,inventory,accountsPayables,otherWorkingCapital,otherNonCashItems,netCashProvidedByOperatingActivities,investmentsInPropertyPlantAndEquipment,acquisitionsNet,purchasesOfInvestments,salesMaturitiesOfInvestments,otherInvestingActivities,netCashProvidedByInvestingActivities,netDebtIssuance,longTermNetDebtIssuance,shortTermNetDebtIssuance,netStockIssuance,netCommonStockIssuance,commonStockIssuance,commonStockRepurchased,netPreferredStockIssuance,netDividendsPaid,commonDividendsPaid,preferredDividendsPaid,otherFinancingActivities,netCashProvidedByFinancingActivities,effectOfForexChangesOnCash,netChangeInCash,cashAtEndOfPeriod,cashAtBeginningOfPeriod,operatingCashFlow,capitalExpenditure,freeCashFlow,incomeTaxesPaid,interestPaid
1699,2025-06-30,AAPG,CNY,0002023311,2025-06-30,2025-06-30 00:00:00,2025,Q2,-1.181536e+09,0.0,0.0,26096000.0,0.0,0.0,0.0,0.0,0.0,1.155440e+09,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0,0.0
1700,2024-12-31,AAPG,CNY,0002023311,2024-12-31,2024-12-31 00:00:00,2024,Q4,-5.684340e+08,45858000.0,0.0,12194000.0,87360000.0,62750000.0,9570000.0,0.0,15040000.0,6.660560e+08,243034000.0,-7762000.0,-9.516000e+06,0.000000e+00,0.000000e+00,-213478000.0,-230756000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,-82138000.0,-82138000.0,10528000.0,0.0,8.931000e+08,9.524340e+08,243034000.0,-7762000.0,235272000.0,0,27400000.0
1701,2024-06-30,AAPG,CNY,0002023311,2024-06-30,2024-06-30 00:00:00,2024,Q2,1.630020e+08,47312000.0,0.0,8730000.0,0.0,0.0,0.0,0.0,0.0,-5.734340e+08,-354390000.0,-16526000.0,0.000000e+00,0.000000e+00,0.000000e+00,-114752000.0,-131278000.0,-98852000.0,-98852000.0,0.0,-1958000.0,-1958000.0,0.0,-1958000.0,0.0,0.0,0.0,0,497716000.0,396906000.0,3150000.0,952434000.0,9.524340e+08,0.000000e+00,-354390000.0,-16526000.0,-370916000.0,0,33156000.0
1702,2023-12-31,AAPG,CNY,0002023311,2023-12-31,2023-12-31 00:00:00,2023,Q4,-5.232860e+08,47620000.0,0.0,13254000.0,-69012000.0,-91536000.0,-6718000.0,0.0,29244000.0,1.738120e+08,-357612000.0,-12132000.0,-2.000000e+07,0.000000e+00,0.000000e+00,118828000.0,86696000.0,0.0,0.0,0.0,-5922000.0,-5922000.0,0.0,-5922000.0,0.0,0.0,0.0,0,-80960000.0,-86882000.0,5858000.0,0.0,1.038048e+09,1.389991e+09,-357612000.0,-12132000.0,-369746000.0,0,37972000.0
1703,2023-06-30,AAPG,CNY,0002023311,2023-06-30,2023-06-30 00:00:00,2023,Q2,-4.095760e+08,45575000.0,0.0,18249000.0,59866000.0,-1000.0,-1000.0,-23114000.0,82980000.0,-8.257900e+07,-368465000.0,-33976000.0,0.000000e+00,0.000000e+00,0.000000e+00,-30798000.0,-64774000.0,-13357000.0,-13357000.0,0.0,470080000.0,470080000.0,470081000.0,-1000.0,0.0,0.0,0.0,0,-1090000.0,455633000.0,21955000.0,-307591000.0,1.038048e+09,1.345639e+09,-368465000.0,-44712000.0,-413175000.0,0,54376000.0
1704,2022-12-31,AAPG,CNY,0002023311,2022-12-31,2022-12-31 00:00:00,2022,Q4,-4.435860e+08,4564420.0,0.0,11052500.0,52206500.0,-194000.0,-2759000.0,12349000.0,42810500.0,2.191200e+07,-326957500.0,-101644000.0,-1.000000e+07,-6.500000e+07,0.000000e+00,-15662000.0,-192306000.0,347937000.0,347937000.0,0.0,-13388000.0,-13388000.0,0.0,-13388000.0,0.0,0.0,0.0,0,-24915000.0,309634000.0,29006000.0,-361247000.0,1.345639e+09,1.706886e+09,-326957500.0,-118481500.0,-445439000.0,0,24924500.0
1705,2022-06-30,AAPG,CNY,0002023311,2022-06-30,2022-06-30 00:00:00,2022,Q2,-4.435860e+08,4564420.0,0.0,11052500.0,52206500.0,-194000.0,-2759000.0,12349000.0,42810500.0,2.191200e+07,-326957500.0,-101644000.0,-1.000000e+07,-6.500000e+07,0.000000e+00,-15662000.0,-192306000.0,347937000.0,347937000.0,0.0,-13388000.0,-13388000.0,0.0,-13388000.0,0.0,0.0,0.0,0,-24915000.0,309634000.0,29006000.0,-361247000.0,1.345639e+09,1.706886e+09,-326957500.0,-118481500.0,-445439000.0,0,24924500.0
1706,2021-12-31,AAPG,CNY,0002023

In [7]:
infotypes = [
    'adjusteddailyprice',
    'unadjusteddailyprice',
    'balancesheet',
    'cashflow',
    'incomestatement',
    'keymetrics',
    'enterprisevalues',
]

i = 0
_df = pd.read_parquet(f"test_data/test_data_{infotypes[i]}.parquet")

_df.head()

,symbol,date,adjOpen,adjHigh,adjLow,adjClose,volume
0,BTMD,2024-12-31,5.88,6.32,5.88,6.18,162100
1,BTMD,2024-12-30,5.96,5.96,5.76,5.91,53500
2,BTMD,2024-12-27,6.21,6.21,5.82,5.99,65622
3,BTMD,2024-12-26,5.91,6.19,5.85,6.17,72126
4,BTMD,2024-12-24,5.79,5.96,5.68,5.96,31300


In [9]:
_df.dtypes

symbol       object
date         object
adjOpen     float64
adjHigh     float64
adjLow      float64
adjClose    float64
volume        int64
dtype: object

In [11]:
from dataPullHelpers import normalize_df_for_parquet


_df2 = normalize_df_for_parquet(_df)
_df2

/home/sagemaker-user/QuantTrading/GetFMPData/dataPullHelpers.py:43: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(non_null, errors="coerce", utc=False)


,symbol,date,adjOpen,adjHigh,adjLow,adjClose,volume
0,BTMD,2024-12-31,5.88,6.32,5.88,6.18,162100
1,BTMD,2024-12-30,5.96,5.96,5.76,5.91,53500
2,BTMD,2024-12-27,6.21,6.21,5.82,5.99,65622
3,BTMD,2024-12-26,5.91,6.19,5.85,6.17,72126
4,BTMD,2024-12-24,5.79,5.96,5.68,5.96,31300
...,...,...,...,...,...,...,...
1455352,DBRG-PI,2017-06-01,12.70,12.71,12.65,12.69,153290
1455353,DBRG-PI,2017-05-31,12.70,12.72,12.63,12.70,412325
1455354,DBRG-PI,2017-05-30,12.68,12.72,12.62,12.67,647810
1455355,DBRG-PI,2017-05-26,12.68,12.70,12.62,12.65,335140


In [12]:
_df2.dtypes

symbol      string[python]
date        datetime64[ns]
adjOpen            float64
adjHigh            float64
adjLow             float64
adjClose           float64
volume               int64
dtype: object

In [51]:
import importlib
import dataPullHelpers
importlib.reload(dataPullHelpers)
from dataPullHelpers import normalize_df_for_parquet

In [68]:
_df = pd.read_parquet(f"data/data_adjusteddailyprice_1.parquet")
_df.head()

,symbol,date,adjOpen,adjHigh,adjLow,adjClose,volume
0,NCPL,2009-12-31,140.0,140.0,140.0,140.0,0
1,NCPL,2009-12-30,182.0,182.0,182.0,182.0,1
2,NCPL,2009-12-29,182.0,182.0,182.0,182.0,0
3,NCPL,2009-12-28,140.0,140.0,140.0,140.0,0
4,NCPL,2009-12-24,140.0,140.0,140.0,140.0,0


In [69]:
_df.shape

(6311212, 7)

In [43]:
samples = []
for row in _df.sample(5).iterrows():
    samples.append(row[1].to_dict())

samples

[{'symbol': 'ALNY',
  'date': '2007-03-27',
  'adjOpen': 17.26,
  'adjHigh': 17.56,
  'adjLow': 17.25,
  'adjClose': 17.52,
  'volume': 244066},
 {'symbol': 'ACER',
  'date': '2006-09-26',
  'adjOpen': 9533.32,
  'adjHigh': 9533.32,
  'adjLow': 8706.68,
  'adjClose': 9266.68,
  'volume': 5},
 {'symbol': 'WBS',
  'date': '2002-10-23',
  'adjOpen': 18.4,
  'adjHigh': 18.48,
  'adjLow': 18.1,
  'adjClose': 18.48,
  'volume': 343900},
 {'symbol': 'CNX',
  'date': '2000-08-28',
  'adjOpen': 6.06,
  'adjHigh': 6.22,
  'adjLow': 5.98,
  'adjClose': 6.2,
  'volume': 216960},
 {'symbol': 'OPCH',
  'date': '2005-04-12',
  'adjOpen': 23.6,
  'adjHigh': 23.88,
  'adjLow': 23.12,
  'adjClose': 23.6,
  'volume': 37775}]

In [57]:
samples[0]['adjOpen'] = 279855000000000000

In [58]:
samples

[{'symbol': 'ALNY',
  'date': '2007-03-27',
  'adjOpen': 279855000000000000,
  'adjHigh': 17.56,
  'adjLow': 17.25,
  'adjClose': 17.52,
  'volume': 244066},
 {'symbol': 'ACER',
  'date': '2006-09-26',
  'adjOpen': 9533.32,
  'adjHigh': 9533.32,
  'adjLow': 8706.68,
  'adjClose': 9266.68,
  'volume': 5},
 {'symbol': 'WBS',
  'date': '2002-10-23',
  'adjOpen': 18.4,
  'adjHigh': 18.48,
  'adjLow': 18.1,
  'adjClose': 18.48,
  'volume': 343900},
 {'symbol': 'CNX',
  'date': '2000-08-28',
  'adjOpen': 6.06,
  'adjHigh': 6.22,
  'adjLow': 5.98,
  'adjClose': 6.2,
  'volume': 216960},
 {'symbol': 'OPCH',
  'date': '2005-04-12',
  'adjOpen': 23.6,
  'adjHigh': 23.88,
  'adjLow': 23.12,
  'adjClose': 23.6,
  'volume': 37775}]

In [64]:
_df2 = pd.DataFrame(samples)
_df2["adjOpen"] = _df2["adjOpen"].astype(object)  # force object
_df2.loc[0, "adjOpen"] = 279855000000000000     # ensure it's a Python int
_df2

,symbol,date,adjOpen,adjHigh,adjLow,adjClose,volume
0,ALNY,2007-03-27,279855000000000000,17.56,17.25,17.52,244066
1,ACER,2006-09-26,9533.32,9533.32,8706.68,9266.68,5
2,WBS,2002-10-23,18.4,18.48,18.10,18.48,343900
3,CNX,2000-08-28,6.06,6.22,5.98,6.20,216960
4,OPCH,2005-04-12,23.6,23.88,23.12,23.60,37775


In [65]:
_df2.dtypes

symbol       object
date         object
adjOpen      object
adjHigh     float64
adjLow      float64
adjClose    float64
volume        int64
dtype: object

In [66]:
_df2 = normalize_df_for_parquet(_df2, treat_huge_int_as_string=False)
_df2

,symbol,date,adjOpen,adjHigh,adjLow,adjClose,volume
0,ALNY,2007-03-27,2.798550e+17,17.56,17.25,17.52,244066
1,ACER,2006-09-26,9.533320e+03,9533.32,8706.68,9266.68,5
2,WBS,2002-10-23,1.840000e+01,18.48,18.10,18.48,343900
3,CNX,2000-08-28,6.060000e+00,6.22,5.98,6.20,216960
4,OPCH,2005-04-12,2.360000e+01,23.88,23.12,23.60,37775


In [67]:
_df2.dtypes

symbol      string[python]
date        datetime64[ns]
adjOpen            float64
adjHigh            float64
adjLow             float64
adjClose           float64
volume               int64
dtype: object

In [ ]:
_df = pd.read_parquet(f"s3://sagemaker-us-east-1-209479286572/datasets/QuantTradingModelData/raw/20251229/data_unadjusteddailyprice_tk0_pd3.parquet")
_df.head()

In [12]:
_df.dtypes

symbol      string[python]
date        datetime64[ns]
adjOpen            float64
adjHigh            float64
adjLow             float64
adjClose           float64
volume             float64
dtype: object

In [13]:
_df.shape

(9613995, 7)

In [82]:
# _df2 = normalize_df_for_parquet(_df, treat_huge_int_as_string=False)

In [83]:
# _df2.dtypes

In [84]:
# _df2.head()

In [85]:
# _df2.shape

In [89]:
# _df2.to_parquet(
#     "data/data_adjusteddailyprice_tk0_pd1.parquet",
#     compression='zstd',
#     index=False
# )

In [14]:
import gc

gc.collect()

15

In [1]:
!ls

DataEval.ipynb	  FMPDataPull.py  __pycache__  dataPullHelpers.py  raw_data
FMPDataClient.py  __init__.py	  data	       error_log.json	   test_data


In [2]:
import json

with open("raw_data/raw_data_temp.json", "r") as f:
    raw_data = json.load(f)

In [5]:
raw_data['SNES'].keys()

dict_keys(['balance sheet', 'cash flow', 'key metrics', 'enterprise values'])

In [12]:
raw_data_df = pd.DataFrame(raw_data['SNES']['income statement'])
raw_data_df.head()

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,revenue,costOfRevenue,grossProfit,researchAndDevelopmentExpenses,generalAndAdministrativeExpenses,sellingAndMarketingExpenses,sellingGeneralAndAdministrativeExpenses,otherExpenses,operatingExpenses,costAndExpenses,netInterestIncome,interestIncome,interestExpense,depreciationAndAmortization,ebitda,ebit,nonOperatingIncomeExcludingInterest,operatingIncome,totalOtherIncomeExpensesNet,incomeBeforeTax,incomeTaxExpense,netIncomeFromContinuingOperations,netIncomeFromDiscontinuedOperations,otherAdjustmentsToNetIncome,netIncome,netIncomeDeductions,bottomLineNetIncome,eps,epsDiluted,weightedAverageShsOut,weightedAverageShsOutDil
0,2025-09-30,SNES,USD,0001680378,2025-11-10,2025-11-10 17:28:02,2025,Q3,690000,257000,433000,400000,0,0,0,1380000,1780000,2037000,49000,55000,6000,29000,-1263000,-1292000,-55000,-1347000,49000,-1298000,0,-1298000,0,0,-1298000,0.0,-1298000,-0.28,-0.28,4668009.0,4668009.0
1,2025-06-30,SNES,USD,0001680378,2025-08-08,2025-08-07 17:36:07,2025,Q2,625000,216000,409000,427000,1532000,64000,1596000,0,2023000,2239000,-2000,4000,6000,35000,-1575000,-1610000,-4000,-1614000,-2000,-1616000,0,-1616000,0,0,-1616000,0.0,-1616000,-0.87,-0.87,1854531.0,1854531.0
2,2025-03-31,SNES,USD,0001680378,2025-05-08,2025-05-08 17:21:24,2025,Q1,485000,172000,313000,418000,1494000,64000,1558000,0,1976000,2148000,-2000,3000,5000,39000,-1621000,-1660000,-3000,-1663000,-2000,-1665000,0,-1665000,0,0,-1665000,0.0,-1665000,-1.28,-1.28,1299971.0,1299971.0
3,2024-12-31,SNES,USD,0001680378,2025-03-13,2025-03-12 17:38:15,2024,Q4,501000,196000,305000,424000,1074000,64000,1138000,0,1562000,1758000,1000,8000,7000,41000,-1207000,-1248000,-9000,-1257000,2000,-1255000,0,-1255000,0,0,-1255000,0.0,-1255000,-2.07,-2.07,729400.0,729400.0
4,2024-09-30,SNES,USD,0001680378,2024-11-12,2024-11-12 17:23:16,2024,Q3,482000,167000,315000,451000,1331000,80000,1411000,0,1862000,2029000,5000,11000,6000,42000,-1465000,-1507000,-40000,-1547000,34000,-1513000,0,-1513000,0,0,-1513000,0.0,-1513000,-2.07,-2.07,729400.0,729400.0


In [13]:
raw_data_df.shape

(43, 39)

In [6]:
from collections import defaultdict
from dataPullHelpers import normalize_df_for_parquet


_datasets, errorLog = defaultdict(list), defaultdict(dict)
for tk, _val in raw_data.items():
    for info, _data in _val.items():
        if isinstance(_data, dict) and '__error__' in _data:
            errorLog[tk][info] = _data['__error__']
        else:
            df_new = pd.DataFrame(_data)
            df_new = normalize_df_for_parquet(
                df_new,
                datetime_success_ratio=0.9,
                numeric_success_ratio=0.9,
                treat_huge_int_as_string=False
            )
            df_new = df_new.dropna(axis=1, how="all")
            if df_new.empty:
                continue
            _datasets[info].append(df_new)

_datasets_final = {}
for info, dfs in _datasets.items():
    if not dfs:
        continue
    _datasets_final[info] = pd.concat(dfs, ignore_index=True)

In [55]:
_datasets_final['income statement'].dtypes

date                                       datetime64[ns]
symbol                                     string[python]
reportedCurrency                           string[python]
cik                                        string[python]
filingDate                                 datetime64[ns]
acceptedDate                               datetime64[ns]
fiscalYear                                        float64
period                                     string[python]
revenue                                           float64
costOfRevenue                                     float64
grossProfit                                       float64
researchAndDevelopmentExpenses                    float64
generalAndAdministrativeExpenses                  float64
sellingAndMarketingExpenses                       float64
sellingGeneralAndAdministrativeExpenses           float64
otherExpenses                                     float64
operatingExpenses                                 float64
costAndExpense

In [56]:
# Legacy local write — raw snapshots are now produced by fmp_data_pull.py under raw/<label>/ on S3
_datasets_final['income statement'].to_parquet(
    "data/data_incomestatement_tk0_pd1.parquet",
    compression='zstd',
    index=False
)

In [8]:
# Legacy local write — raw snapshots are now produced by fmp_data_pull.py under raw/<label>/ on S3
for info, _dataset in _datasets_final.items():
    _dataset.to_parquet(
        f"data/data_{info.replace(' ', '')}_tk0_pd1.parquet",
        compression='zstd',
        index=False
    )

In [50]:
symbol = 'TDIC'

price1 = getFMPData('historical-price-eod/dividend-adjusted', symbol=symbol, from_='2000-01-01', to='2025-12-31')
price2 = getFMPData('historical-price-eod/non-split-adjusted', symbol=symbol, from_='2000-01-01', to='2025-12-31')
bal = getFMPData('balance-sheet-statement', symbol=symbol, limit=120, period='quarter')
incm = getFMPData('income-statement', symbol=symbol, limit=120, period='quarter')
cf = getFMPData('cash-flow-statement', symbol=symbol, limit=120, period='quarter')
ev = getFMPData('enterprise-values', symbol=symbol, limit=120, period='quarter')
km = getFMPData('key-metrics', symbol=symbol, limit=120, period='quarter')
len(price1), len(price2), len(bal), len(cf), len(incm), len(km), len(ev)

(110, 110, 3, 0, 0, 3, 3)

In [9]:
_datasets_final.keys()

dict_keys(['balance sheet', 'cash flow', 'key metrics', 'enterprise values'])

In [12]:
_datasets_final['balance sheet']

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,cashAndCashEquivalents,shortTermInvestments,cashAndShortTermInvestments,netReceivables,accountsReceivables,otherReceivables,inventory,prepaids,otherCurrentAssets,totalCurrentAssets,propertyPlantEquipmentNet,goodwill,intangibleAssets,goodwillAndIntangibleAssets,longTermInvestments,taxAssets,otherNonCurrentAssets,totalNonCurrentAssets,otherAssets,totalAssets,totalPayables,accountPayables,otherPayables,accruedExpenses,shortTermDebt,capitalLeaseObligationsCurrent,taxPayables,deferredRevenue,otherCurrentLiabilities,totalCurrentLiabilities,longTermDebt,capitalLeaseObligationsNonCurrent,deferredRevenueNonCurrent,deferredTaxLiabilitiesNonCurrent,otherNonCurrentLiabilities,totalNonCurrentLiabilities,otherLiabilities,capitalLeaseObligations,totalLiabilities,treasuryStock,preferredStock,commonStock,retainedEarnings,additionalPaidInCapital,accumulatedOtherComprehensiveIncomeLoss,otherTotalStockholdersEquity,totalStockholdersEquity,totalEquity,minorityInterest,totalLiabilitiesAndTotalEquity,totalInvestments,totalDebt,netDebt
0,2025-09-30,SNES,USD,0001680378,2025-11-10,2025-11-10 17:28:02,2025.0,Q3,7.278000e+06,2970000.0,1.024800e+07,4.540000e+05,454000.0,0.000000e+00,767000.0,313000.0,0.0,1.178200e+07,2806000.0,0.0,0.0,0.0,0.0,0.0,3.600000e+04,2.842000e+06,0.0,1.462400e+07,2.380000e+05,2.380000e+05,0.0,333000.0,60000.0,95000.0,0.0,0.0,2.200000e+04,7.480000e+05,160000.0,2368000.0,0.0,0.0,0.0,2.528000e+06,0.0,2463000.0,3.276000e+06,0.0,0.0,5000.0,-140676000.0,152019000.0,-2.842171e-08,2.842171e-08,1.134800e+07,1.134800e+07,0.0,1.462400e+07,2970000.0,2.683000e+06,-4595000.0
1,2025-06-30,SNES,USD,0001680378,2025-08-08,2025-08-07 17:36:07,2025.0,Q2,6.055000e+06,0.0,6.055000e+06,4.700000e+05,470000.0,0.000000e+00,729000.0,39000.0,181000.0,7.474000e+06,2833000.0,0.0,0.0,0.0,0.0,0.0,5.800000e+04,2.891000e+06,0.0,1.036500e+07,1.230000e+05,1.230000e+05,0.0,310000.0,59000.0,52000.0,0.0,12000.0,2.540000e+05,8.100000e+05,175000.0,2403000.0,0.0,0.0,0.0,2.578000e+06,0.0,2455000.0,3.388000e+06,0.0,0.0,4000.0,-139378000.0,146351000.0,0.000000e+00,0.000000e+00,6.977000e+06,6.977000e+06,0.0,1.036500e+07,0.0,2.689000e+06,-3366000.0
2,2025-03-31,SNES,USD,0001680378,2025-05-08,2025-05-08 17:21:24,2025.0,Q1,1.655000e+06,0.0,1.655000e+06,4.980000e+05,498000.0,0.000000e+00,753000.0,37000.0,221000.0,3.164000e+06,404000.0,0.0,0.0,0.0,0.0,0.0,5.800000e+04,4.620000e+05,0.0,3.626000e+06,1.640000e+05,1.640000e+05,0.0,33000.0,57000.0,0.0,0.0,12000.0,2.770000e+05,5.430000e+05,191000.0,0.0,0.0,0.0,0.0,1.910000e+05,0.0,0.0,7.340000e+05,0.0,0.0,2000.0,-137762000.0,140652000.0,0.000000e+00,0.000000e+00,2.892000e+06,2.892000e+06,0.0,3.626000e+06,0.0,2.480000e+05,-1407000.0
3,2024-12-31,SNES,USD,0001680378,2025-03-13,2025-03-12 17:38:15,2024.0,Q4,1.307000e+06,0.0,1.307000e+06,3.350000e+05,335000.0,0.000000e+00,794000.0,27000.0,350000.0,2.813000e+06,407000.0,0.0,0.0,0.0,0.0,0.0,5.800000e+04,4.650000e+05,0.0,3.278000e+06,2.150000e+05,2.150000e+05,0.0,34000.0,56000.0,0.0,0.0,12000.0,2.440000e+05,5.610000e+05,206000.0,0.0,0.0,0.0,0.0,2.060000e+05,0.0,0.0,7.670000e+05,0.0,0.0,1000.0,-136097000.0,138607000.0,0.000000e+00,0.000000e+00,2.511000e+06,2.511000e+06,0.0,3.278000e+06,0.0,2.620000e+05,-1045000.0
4,2024-09-30,SNES,USD,0001680378,2024-11-12,2024-11-12 17:23:16,2024.0,Q3,2.518000e+06,0.0,2.518000e+06,2.140000e+05,214000.0,0.000000e+00,880000.0,52000.0,308000.0,3.972000e+06,419000.0,0.0,0.0,0.0,0.0,0.0,5.800000e+04,4.770000e+05,0.0,4.449000e+06,1.280000e+05,1.280000e+05,0.0,17000.0,53000.0,41000.0,0.0,12000.0,3.770000e+05,6.280000e+05,170000.0,0.0,0.0,0.0,0.0,1.700000e+05,0.0,41000.0,7.980000e+05,0.0,0.0,1000.0,-134842000.0,138492000.0,0.000000e+00,0.000000e+00,3.651000e+06,3.651000e+06,0.0,4.449000e+06,0.0,2.640000e+05,-2254000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.